# 🥇 Silver → Gold

Cria um modelo dimensional simples (Star Schema) e agregações a partir da Silver.

In [ ]:
import os
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'SBD2', cwd.parent]
    for root in candidates:
        if (root / 'data').exists() and (root / 'notebooks').exists():
            return root
    return cwd

PROJECT_ROOT = find_project_root()
SILVER_DIR = PROJECT_ROOT / 'data' / 'silver'
GOLD_DIR = PROJECT_ROOT / 'data' / 'gold'
GOLD_DIR.mkdir(parents=True, exist_ok=True)

BATCH_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

print('📁 Projeto:', PROJECT_ROOT)
print('📁 Silver:', SILVER_DIR)
print('📁 Gold  :', GOLD_DIR)
print('🔖 Batch :', BATCH_ID)

In [ ]:
crimes_path = SILVER_DIR / 'crimes.parquet'
if not crimes_path.exists():
    raise FileNotFoundError(f'Arquivo Silver não encontrado: {crimes_path}')

df = pd.read_parquet(crimes_path)
print('✅ Silver carregada', df.shape)
df.head()

In [ ]:
# Dimensão Data
date_series = pd.to_datetime(df['date_occurred'], errors='coerce')
min_date = date_series.min()
max_date = date_series.max()
date_range = pd.date_range(min_date, max_date, freq='D')
dim_date = pd.DataFrame({'full_date': date_range})
dim_date['sk_date'] = np.arange(1, len(dim_date) + 1)
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['day'] = dim_date['full_date'].dt.day
dim_date['day_of_week'] = dim_date['full_date'].dt.dayofweek
dim_date['is_weekend'] = dim_date['day_of_week'].isin([5,6])

# Dimensão Tempo (hora)
dim_time = pd.DataFrame({'hour': list(range(24))})
dim_time['sk_time'] = np.arange(1, 25)

print('✅ dim_date:', len(dim_date), ' | dim_time:', len(dim_time))

In [ ]:
# Dimensões (básicas)
dim_area = df[['area_code','area_name']].drop_duplicates().sort_values('area_code').reset_index(drop=True)
dim_area['sk_area'] = np.arange(1, len(dim_area) + 1)

dim_crime_type = df[['crime_code','crime_description','is_violent']].drop_duplicates().sort_values('crime_code').reset_index(drop=True)
dim_crime_type['sk_crime_type'] = np.arange(1, len(dim_crime_type) + 1)

dim_weapon = df[['weapon_code','weapon_description']].drop_duplicates().reset_index(drop=True)
dim_weapon = dim_weapon.dropna(subset=['weapon_code']).sort_values('weapon_code').reset_index(drop=True)
dim_weapon['sk_weapon'] = np.arange(1, len(dim_weapon) + 1)

dim_premise = df[['premise_code','premise_description']].drop_duplicates().reset_index(drop=True)
dim_premise = dim_premise.dropna(subset=['premise_code']).sort_values('premise_code').reset_index(drop=True)
dim_premise['sk_premise'] = np.arange(1, len(dim_premise) + 1)

# Vítima (perfil mínimo)
vict_cols = [c for c in ['victim_sex','victim_descent','victim_age'] if c in df.columns]
dim_victim = df[vict_cols].drop_duplicates().reset_index(drop=True)
dim_victim['sk_victim'] = np.arange(1, len(dim_victim) + 1)

print('✅ dims criadas')

In [ ]:
# Fato: resolve SKs
tmp = df.copy()
tmp['full_date'] = pd.to_datetime(tmp['date_occurred'], errors='coerce').dt.normalize()
tmp['hour'] = pd.to_numeric(tmp.get('hour_occurred'), errors='coerce')

fato = (tmp
  .merge(dim_date[['sk_date','full_date']], on='full_date', how='left')
  .merge(dim_time[['sk_time','hour']], on='hour', how='left')
  .merge(dim_area[['sk_area','area_code']], on='area_code', how='left')
  .merge(dim_crime_type[['sk_crime_type','crime_code']], on='crime_code', how='left')
  .merge(dim_weapon[['sk_weapon','weapon_code']], on='weapon_code', how='left')
  .merge(dim_premise[['sk_premise','premise_code']], on='premise_code', how='left')
)

# Join vítima (chave natural simples)
if len(dim_victim.columns) > 1:
    fato = fato.merge(dim_victim, on=vict_cols, how='left')
else:
    fato['sk_victim'] = 1

fato_crimes = fato[[
    'crime_id','sk_date','sk_time','sk_area','sk_crime_type','sk_weapon','sk_premise','sk_victim'
]].copy()

print('✅ fato_crimes:', fato_crimes.shape)
fato_crimes.head()

In [ ]:
# Agregações (exemplos)
agg_area_month = (
    df.groupby(['area_name','year_occurred','month_occurred'])
      .size().reset_index(name='total_crimes')
)
agg_type_year = (
    df.groupby(['crime_description','year_occurred'])
      .size().reset_index(name='total_crimes')
)

print('✅ aggs:', len(agg_area_month), len(agg_type_year))

In [ ]:
# Persistência Gold
dim_date.to_parquet(GOLD_DIR / 'dim_date.parquet', index=False)
dim_time.to_parquet(GOLD_DIR / 'dim_time.parquet', index=False)
dim_area.to_parquet(GOLD_DIR / 'dim_area.parquet', index=False)
dim_crime_type.to_parquet(GOLD_DIR / 'dim_crime_type.parquet', index=False)
dim_weapon.to_parquet(GOLD_DIR / 'dim_weapon.parquet', index=False)
dim_premise.to_parquet(GOLD_DIR / 'dim_premise.parquet', index=False)
dim_victim.to_parquet(GOLD_DIR / 'dim_victim.parquet', index=False)

fato_crimes.to_parquet(GOLD_DIR / 'fato_crimes.parquet', index=False)
agg_area_month.to_parquet(GOLD_DIR / 'agg_area_month.parquet', index=False)
agg_type_year.to_parquet(GOLD_DIR / 'agg_type_year.parquet', index=False)

meta = {
  'batch_id': BATCH_ID,
  'transformation_timestamp': datetime.now().isoformat(),
  'source_layer': 'silver',
  'target_layer': 'gold',
  'fact_records': int(len(fato_crimes)),
}
meta_path = GOLD_DIR / f'gold_metadata_{BATCH_ID}.json'
with open(meta_path, 'w', encoding='utf-8') as f:
  json.dump(meta, f, indent=2, default=str)

print('✅ Gold salvo em', GOLD_DIR)
print('📁 Arquivos:')
for p in sorted(GOLD_DIR.iterdir()):
  if p.is_file():
    print(' -', p.name)

# 🥇 Silver → Gold
## Crime Data from 2020 to Present - Los Angeles

### Arquitetura Medalhão - Camada Gold

Este notebook realiza a transformação dos dados da camada Silver para a camada Gold.

**Camada Gold**: Modelo dimensional (Star Schema) otimizado para análises, relatórios e dashboards.

### Estrutura:
1. Tabelas de Dimensão (DIM)
2. Tabela Fato (FACT)
3. Tabelas Agregadas (AGG)
4. Views para Analytics

In [ ]:
# Importações
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

# Configurações
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print('✅ Bibliotecas carregadas com sucesso!')
print(f'📅 Data de execução: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## 1. Configuração de Caminhos

In [ ]:
# Configuração de diretórios (robusto a diferentes CWDs)
from pathlib import Path

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / 'SBD2', cwd.parent]
    for root in candidates:
        if (root / 'data').exists() and (root / 'notebooks').exists():
            return root
        if (root / 'src').exists() and (root / 'notebooks').exists():
            return root
    return cwd

PROJECT_ROOT = find_project_root()
SILVER_DATA_DIR = PROJECT_ROOT / 'data' / 'silver'
GOLD_DATA_DIR = PROJECT_ROOT / 'data' / 'gold'

# Criar diretório Gold se não existir
GOLD_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Parâmetros de transformação
BATCH_ID = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f'📁 Projeto: {PROJECT_ROOT}')
print(f'📁 Silver: {SILVER_DATA_DIR}')
print(f'📁 Gold: {GOLD_DATA_DIR}')
print(f'🔖 Batch ID: {BATCH_ID}')

## 2. Carregamento dos Dados Silver

In [ ]:
# Carregar tabela principal de crimes
crimes_path = SILVER_DATA_DIR / 'crimes.parquet'
if not crimes_path.exists():
    raise FileNotFoundError(f"Arquivo Silver não encontrado: {crimes_path}")

try:
    df_silver = pd.read_parquet(crimes_path)
except Exception as e:
    raise RuntimeError(
        "Falha ao ler Parquet da camada Silver. "
        "Instale 'pyarrow' (recomendado) ou 'fastparquet'. "
        f"Erro: {e}"
    )

print(f"✅ Dados Silver carregados!")
print(f"📊 Shape: {df_silver.shape[0]:,} linhas x {df_silver.shape[1]} colunas")
print(f"💾 Memória: {df_silver.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Carregar dimensões Silver
dim_areas_silver = pd.read_parquet(SILVER_DATA_DIR / 'dim_areas.parquet')
dim_crime_types_silver = pd.read_parquet(SILVER_DATA_DIR / 'dim_crime_types.parquet')
dim_weapons_silver = pd.read_parquet(SILVER_DATA_DIR / 'dim_weapons.parquet')
dim_premises_silver = pd.read_parquet(SILVER_DATA_DIR / 'dim_premises.parquet')

print("✅ Dimensões Silver carregadas!")
print(f"   📊 dim_areas: {len(dim_areas_silver)} registros")
print(f"   📊 dim_crime_types: {len(dim_crime_types_silver)} registros")
print(f"   📊 dim_weapons: {len(dim_weapons_silver)} registros")
print(f"   📊 dim_premises: {len(dim_premises_silver)} registros")

In [ ]:
# Visualizar amostra dos dados Silver
df_silver.head()

## 3. Criação das Dimensões Gold (Star Schema)

### 3.1 Dimensão: Data (dim_date)

In [ ]:
# Criar dimensão de data
print("📅 Criando dimensão de data...")

# Obter range de datas
date_col = pd.to_datetime(df_silver['date_occurred'])
min_date = date_col.min()
max_date = date_col.max()

# Gerar todas as datas no range
date_range = pd.date_range(start=min_date, end=max_date, freq='D')

dim_date = pd.DataFrame({'full_date': date_range})
dim_date['sk_date'] = range(1, len(dim_date) + 1)  # Surrogate Key
dim_date['year'] = dim_date['full_date'].dt.year
dim_date['quarter'] = dim_date['full_date'].dt.quarter
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['month_name'] = dim_date['full_date'].dt.month_name()
dim_date['week_of_year'] = dim_date['full_date'].dt.isocalendar().week
dim_date['day_of_month'] = dim_date['full_date'].dt.day
dim_date['day_of_week'] = dim_date['full_date'].dt.dayofweek
dim_date['day_name'] = dim_date['full_date'].dt.day_name()
dim_date['is_weekend'] = dim_date['day_of_week'].isin([5, 6])
dim_date['is_holiday'] = False  # Pode ser enriquecido com calendário de feriados

# Reordenar colunas
dim_date = dim_date[['sk_date', 'full_date', 'year', 'quarter', 'month', 'month_name',
                     'week_of_year', 'day_of_month', 'day_of_week', 'day_name', 
                     'is_weekend', 'is_holiday']]

print(f"✅ dim_date criada: {len(dim_date)} registros")
print(f"   📅 Período: {min_date.date()} a {max_date.date()}")
dim_date.head(10)

### 3.2 Dimensão: Tempo (dim_time)

In [ ]:
# Criar dimensão de tempo (24 horas)
print("🕐 Criando dimensão de tempo...")

hours = list(range(24))
dim_time = pd.DataFrame({'hour': hours})
dim_time['sk_time'] = range(1, len(dim_time) + 1)  # Surrogate Key
dim_time['minute'] = 0  # Simplificado - apenas horas

# Período do dia
def get_period(hour):
    if 0 <= hour < 6:
        return 'MADRUGADA'
    elif 6 <= hour < 12:
        return 'MANHA'
    elif 12 <= hour < 18:
        return 'TARDE'
    else:
        return 'NOITE'

dim_time['period_of_day'] = dim_time['hour'].apply(get_period)

# Horário de pico (rush hour: 7-9 e 17-19)
dim_time['is_rush_hour'] = dim_time['hour'].isin([7, 8, 9, 17, 18, 19])

# Reordenar colunas
dim_time = dim_time[['sk_time', 'hour', 'minute', 'period_of_day', 'is_rush_hour']]

print(f"✅ dim_time criada: {len(dim_time)} registros")
dim_time

### 3.3 Dimensão: Área (dim_area)

In [ ]:
# Criar dimensão de área
print("🗺️ Criando dimensão de área...")

dim_area = dim_areas_silver.copy()
dim_area['sk_area'] = range(1, len(dim_area) + 1)  # Surrogate Key

# Adicionar região (baseado no código da área)
def get_region(area_name):
    north_areas = ['DEVONSHIRE', 'FOOTHILL', 'MISSION', 'NORTH HOLLYWOOD', 'VAN NUYS', 'WEST VALLEY']
    south_areas = ['77TH STREET', 'HARBOR', 'SOUTHEAST', 'SOUTHWEST']
    central_areas = ['CENTRAL', 'HOLLENBECK', 'RAMPART']
    west_areas = ['HOLLYWOOD', 'OLYMPIC', 'PACIFIC', 'WEST LA', 'WILSHIRE']
    
    if area_name in north_areas:
        return 'NORTH'
    elif area_name in south_areas:
        return 'SOUTH'
    elif area_name in central_areas:
        return 'CENTRAL'
    elif area_name in west_areas:
        return 'WEST'
    else:
        return 'OTHER'

dim_area['region'] = dim_area['area_name'].apply(get_region)

# Reordenar colunas
dim_area = dim_area[['sk_area', 'area_code', 'area_name', 'region', 'total_crimes']]

print(f"✅ dim_area criada: {len(dim_area)} registros")
dim_area

### 3.4 Dimensão: Tipo de Crime (dim_crime_type)

In [ ]:
# Criar dimensão de tipo de crime
print("🔍 Criando dimensão de tipo de crime...")

dim_crime_type = dim_crime_types_silver.copy()
dim_crime_type['sk_crime_type'] = range(1, len(dim_crime_type) + 1)  # Surrogate Key

# Adicionar categoria e nível de severidade
def get_crime_category(desc):
    if pd.isna(desc):
        return 'OTHER'
    desc = str(desc).upper()
    if 'THEFT' in desc or 'STOLEN' in desc or 'BURGLARY' in desc:
        return 'THEFT'
    elif 'ASSAULT' in desc or 'BATTERY' in desc:
        return 'ASSAULT'
    elif 'VEHICLE' in desc:
        return 'VEHICLE'
    elif 'VANDALISM' in desc:
        return 'VANDALISM'
    elif 'ROBBERY' in desc:
        return 'ROBBERY'
    elif 'RAPE' in desc or 'SEX' in desc:
        return 'SEX_CRIME'
    elif 'HOMICIDE' in desc or 'MURDER' in desc:
        return 'HOMICIDE'
    elif 'FRAUD' in desc or 'FORGERY' in desc:
        return 'FRAUD'
    elif 'WEAPON' in desc or 'GUN' in desc:
        return 'WEAPONS'
    elif 'DRUG' in desc or 'NARCOTIC' in desc:
        return 'DRUGS'
    else:
        return 'OTHER'

def get_severity_level(is_violent):
    return 3 if is_violent else 1

dim_crime_type['crime_category'] = dim_crime_type['crime_description'].apply(get_crime_category)
dim_crime_type['severity_level'] = dim_crime_type['is_violent'].apply(get_severity_level)

# Reordenar colunas
dim_crime_type = dim_crime_type[['sk_crime_type', 'crime_code', 'crime_description', 
                                  'crime_category', 'is_violent', 'severity_level', 
                                  'total_occurrences']]

print(f"✅ dim_crime_type criada: {len(dim_crime_type)} registros")
print(f"\n📊 Distribuição por categoria:")
print(dim_crime_type['crime_category'].value_counts())

### 3.5 Dimensão: Arma (dim_weapon)

In [ ]:
# Criar dimensão de arma
print("🔫 Criando dimensão de arma...")

dim_weapon = dim_weapons_silver.copy()
dim_weapon['sk_weapon'] = range(1, len(dim_weapon) + 1)  # Surrogate Key

# Adicionar nível de letalidade
def get_lethality(category):
    if category == 'ARMA_DE_FOGO':
        return 5
    elif category == 'ARMA_BRANCA':
        return 4
    elif category == 'VEICULO':
        return 3
    elif category == 'FORCA_CORPORAL':
        return 2
    elif category == 'SEM_ARMA':
        return 0
    else:
        return 1

dim_weapon['lethality_level'] = dim_weapon['weapon_category'].apply(get_lethality)

# Reordenar colunas
dim_weapon = dim_weapon[['sk_weapon', 'weapon_code', 'weapon_description', 
                          'weapon_category', 'lethality_level']]

print(f"✅ dim_weapon criada: {len(dim_weapon)} registros")
dim_weapon.head(10)

### 3.6 Dimensão: Local (dim_premise)

In [ ]:
# Criar dimensão de local
print("🏢 Criando dimensão de local...")

dim_premise = dim_premises_silver.copy()
dim_premise['sk_premise'] = range(1, len(dim_premise) + 1)  # Surrogate Key

# Adicionar categoria e flag de local público
def get_premise_category(desc):
    if pd.isna(desc):
        return 'OTHER'
    desc = str(desc).upper()
    if 'STREET' in desc or 'SIDEWALK' in desc or 'PARKING' in desc:
        return 'OUTDOOR'
    elif 'RESIDENCE' in desc or 'APARTMENT' in desc or 'HOUSE' in desc or 'HOME' in desc:
        return 'RESIDENTIAL'
    elif 'STORE' in desc or 'SHOP' in desc or 'RETAIL' in desc:
        return 'RETAIL'
    elif 'OFFICE' in desc or 'BUILDING' in desc:
        return 'COMMERCIAL'
    elif 'SCHOOL' in desc or 'COLLEGE' in desc:
        return 'EDUCATIONAL'
    elif 'RESTAURANT' in desc or 'BAR' in desc or 'HOTEL' in desc:
        return 'HOSPITALITY'
    elif 'VEHICLE' in desc or 'CAR' in desc:
        return 'VEHICLE'
    else:
        return 'OTHER'

def is_public(desc):
    if pd.isna(desc):
        return False
    desc = str(desc).upper()
    public_keywords = ['STREET', 'SIDEWALK', 'PARK', 'PUBLIC', 'TRANSIT', 'BUS', 'TRAIN']
    return any(kw in desc for kw in public_keywords)

dim_premise['premise_category'] = dim_premise['premise_description'].apply(get_premise_category)
dim_premise['is_public'] = dim_premise['premise_description'].apply(is_public)

# Reordenar colunas
dim_premise = dim_premise[['sk_premise', 'premise_code', 'premise_description', 
                            'premise_category', 'is_public']]

print(f"✅ dim_premise criada: {len(dim_premise)} registros")
print(f"\n📊 Distribuição por categoria:")
print(dim_premise['premise_category'].value_counts())

### 3.7 Dimensão: Vítima (dim_victim)

In [ ]:
# Criar dimensão de vítima (perfil demográfico)
print("👤 Criando dimensão de vítima...")

# Obter combinações únicas
victim_cols = ['age_group', 'victim_sex', 'victim_descent', 'victim_descent_desc']
dim_victim = df_silver[victim_cols].drop_duplicates().reset_index(drop=True)

# Adicionar surrogate key
dim_victim['sk_victim'] = range(1, len(dim_victim) + 1)

# Renomear colunas
dim_victim = dim_victim.rename(columns={
    'victim_sex': 'sex',
    'victim_descent': 'descent',
    'victim_descent_desc': 'descent_description'
})

# Reordenar colunas
dim_victim = dim_victim[['sk_victim', 'age_group', 'sex', 'descent', 'descent_description']]

print(f"✅ dim_victim criada: {len(dim_victim)} registros")
dim_victim.head(10)

## 4. Criação da Tabela Fato

In [ ]:
# Criar mapeamentos para lookups
print("🔗 Criando mapeamentos de chaves...")

# Mapeamento de data
dim_date['full_date_str'] = dim_date['full_date'].dt.strftime('%Y-%m-%d')
date_map = dict(zip(dim_date['full_date_str'], dim_date['sk_date']))

# Mapeamento de hora
time_map = dict(zip(dim_time['hour'], dim_time['sk_time']))

# Mapeamento de área
area_map = dict(zip(dim_area['area_code'], dim_area['sk_area']))

# Mapeamento de tipo de crime
crime_type_map = dict(zip(dim_crime_type['crime_code'], dim_crime_type['sk_crime_type']))

# Mapeamento de arma
weapon_map = dict(zip(dim_weapon['weapon_code'], dim_weapon['sk_weapon']))

# Mapeamento de local
premise_map = dict(zip(dim_premise['premise_code'], dim_premise['sk_premise']))

# Mapeamento de vítima (combinação de atributos)
dim_victim['victim_key'] = dim_victim['age_group'] + '|' + dim_victim['sex'] + '|' + dim_victim['descent']
victim_map = dict(zip(dim_victim['victim_key'], dim_victim['sk_victim']))

print("✅ Mapeamentos criados!")

In [ ]:
# Criar tabela fato
print("📊 Criando tabela fato de crimes...")

fact_crimes = df_silver.copy()

# Converter data para string para lookup
fact_crimes['date_str'] = pd.to_datetime(fact_crimes['date_occurred']).dt.strftime('%Y-%m-%d')

# Aplicar lookups para obter surrogate keys
fact_crimes['sk_date'] = fact_crimes['date_str'].map(date_map)
fact_crimes['sk_time'] = fact_crimes['hour_occurred'].map(time_map)
fact_crimes['sk_area'] = fact_crimes['area_code'].map(area_map)
fact_crimes['sk_crime_type'] = fact_crimes['crime_code'].map(crime_type_map)
fact_crimes['sk_weapon'] = fact_crimes['weapon_code'].map(weapon_map)
fact_crimes['sk_premise'] = fact_crimes['premise_code'].map(premise_map)

# Criar chave de vítima e fazer lookup
fact_crimes['victim_key'] = fact_crimes['age_group'] + '|' + fact_crimes['victim_sex'] + '|' + fact_crimes['victim_descent']
fact_crimes['sk_victim'] = fact_crimes['victim_key'].map(victim_map)

print("✅ Lookups aplicados!")

In [ ]:
# Selecionar colunas da tabela fato
fato_crimes = fact_crimes[[
    'crime_id',        # NK - Natural Key
    'sk_area',         # FK - Dimensão Área
    'sk_crime_type',   # FK - Dimensão Tipo de Crime
    'sk_weapon',       # FK - Dimensão Arma
    'sk_premise',      # FK - Dimensão Local
    'sk_date',         # FK - Dimensão Data
    'sk_time',         # FK - Dimensão Tempo
    'sk_victim',       # FK - Dimensão Vítima
    'latitude',        # Métricas
    'longitude',
    'is_violent'
]].copy()

# Adicionar surrogate key
fato_crimes.insert(0, 'sk_crime', range(1, len(fato_crimes) + 1))

# Renomear crime_id para nk_crime_id
fato_crimes = fato_crimes.rename(columns={'crime_id': 'nk_crime_id'})

print(f"✅ Tabela fato criada: {len(fato_crimes):,} registros")
fato_crimes.head(10)

In [ ]:
# Verificar integridade referencial
print("🔍 Verificando integridade referencial...")

null_checks = {
    'sk_date': fato_crimes['sk_date'].isna().sum(),
    'sk_time': fato_crimes['sk_time'].isna().sum(),
    'sk_area': fato_crimes['sk_area'].isna().sum(),
    'sk_crime_type': fato_crimes['sk_crime_type'].isna().sum(),
    'sk_weapon': fato_crimes['sk_weapon'].isna().sum(),
    'sk_premise': fato_crimes['sk_premise'].isna().sum(),
    'sk_victim': fato_crimes['sk_victim'].isna().sum()
}

print("📊 Valores nulos em chaves estrangeiras:")
for key, count in null_checks.items():
    status = "✅" if count == 0 else "⚠️"
    print(f"   {status} {key}: {count:,}")

## 5. Criação de Tabelas Agregadas

In [ ]:
# Agregação: Crimes por Área e Período
print("📊 Criando agregação: Crimes por Área e Período...")

# Juntar fato com dimensões
agg_base = fato_crimes.merge(dim_date[['sk_date', 'year', 'month']], on='sk_date', how='left')
agg_base = agg_base.merge(dim_time[['sk_time', 'period_of_day']], on='sk_time', how='left')

agg_crimes_area_period = agg_base.groupby(
    ['sk_area', 'year', 'month', 'period_of_day']
).agg(
    total_crimes=('sk_crime', 'count'),
    violent_crimes=('is_violent', 'sum')
).reset_index()

agg_crimes_area_period['non_violent_crimes'] = agg_crimes_area_period['total_crimes'] - agg_crimes_area_period['violent_crimes']

print(f"✅ agg_crimes_area_period: {len(agg_crimes_area_period):,} registros")
agg_crimes_area_period.head(10)

In [ ]:
# Agregação: Crimes por Tipo e Ano
print("📊 Criando agregação: Crimes por Tipo e Ano...")

agg_base2 = fato_crimes.merge(dim_date[['sk_date', 'year', 'is_weekend']], on='sk_date', how='left')

agg_crimes_type_year = agg_base2.groupby(
    ['sk_crime_type', 'year']
).agg(
    total_crimes=('sk_crime', 'count'),
    weekend_crimes=('is_weekend', 'sum')
).reset_index()

agg_crimes_type_year['weekday_crimes'] = agg_crimes_type_year['total_crimes'] - agg_crimes_type_year['weekend_crimes']

print(f"✅ agg_crimes_type_year: {len(agg_crimes_type_year):,} registros")
agg_crimes_type_year.head(10)

In [ ]:
# Agregação: Hotspots Geográficos
print("📊 Criando agregação: Hotspots Geográficos...")

# Criar grid de coordenadas (arredondando para 2 casas decimais)
agg_base3 = fato_crimes.merge(dim_date[['sk_date', 'year']], on='sk_date', how='left')
agg_base3 = agg_base3.dropna(subset=['latitude', 'longitude'])
agg_base3['grid_lat'] = agg_base3['latitude'].round(2)
agg_base3['grid_lon'] = agg_base3['longitude'].round(2)

agg_crime_hotspots = agg_base3.groupby(
    ['grid_lat', 'grid_lon', 'year']
).agg(
    total_crimes=('sk_crime', 'count'),
    violent_crimes=('is_violent', 'sum')
).reset_index()

# Classificar nível de hotspot
def get_hotspot_level(count):
    if count > 1000:
        return 'CRITICAL'
    elif count > 500:
        return 'HIGH'
    elif count > 100:
        return 'MEDIUM'
    else:
        return 'LOW'

agg_crime_hotspots['hotspot_level'] = agg_crime_hotspots['total_crimes'].apply(get_hotspot_level)

print(f"✅ agg_crime_hotspots: {len(agg_crime_hotspots):,} registros")
print(f"\n📊 Distribuição de Hotspots:")
print(agg_crime_hotspots['hotspot_level'].value_counts())

## 6. Persistência na Camada Gold

In [ ]:
# Salvar dimensões
print("💾 Salvando dimensões Gold...")

# Remover colunas auxiliares antes de salvar
dim_date_save = dim_date.drop(columns=['full_date_str'], errors='ignore')
dim_victim_save = dim_victim.drop(columns=['victim_key'], errors='ignore')

dim_date_save.to_parquet(GOLD_DATA_DIR / 'dim_date.parquet', index=False)
dim_time.to_parquet(GOLD_DATA_DIR / 'dim_time.parquet', index=False)
dim_area.to_parquet(GOLD_DATA_DIR / 'dim_area.parquet', index=False)
dim_crime_type.to_parquet(GOLD_DATA_DIR / 'dim_crime_type.parquet', index=False)
dim_weapon.to_parquet(GOLD_DATA_DIR / 'dim_weapon.parquet', index=False)
dim_premise.to_parquet(GOLD_DATA_DIR / 'dim_premise.parquet', index=False)
dim_victim_save.to_parquet(GOLD_DATA_DIR / 'dim_victim.parquet', index=False)

print("✅ Dimensões salvas:")
print(f"   📁 dim_date.parquet ({len(dim_date):,} registros)")
print(f"   📁 dim_time.parquet ({len(dim_time):,} registros)")
print(f"   📁 dim_area.parquet ({len(dim_area):,} registros)")
print(f"   📁 dim_crime_type.parquet ({len(dim_crime_type):,} registros)")
print(f"   📁 dim_weapon.parquet ({len(dim_weapon):,} registros)")
print(f"   📁 dim_premise.parquet ({len(dim_premise):,} registros)")
print(f"   📁 dim_victim.parquet ({len(dim_victim):,} registros)")

In [ ]:
# Salvar tabela fato
print("💾 Salvando tabela fato...")

fato_path = GOLD_DATA_DIR / 'fato_crimes.parquet'
fato_crimes.to_parquet(fato_path, index=False)

print(f"✅ fato_crimes.parquet salva ({len(fato_crimes):,} registros)")
print(f"   💾 Tamanho: {fato_path.stat().st_size / 1024**2:.2f} MB")

In [ ]:
# Salvar tabelas agregadas
print("💾 Salvando tabelas agregadas...")

agg_crimes_area_period.to_parquet(GOLD_DATA_DIR / 'agg_crimes_area_period.parquet', index=False)
agg_crimes_type_year.to_parquet(GOLD_DATA_DIR / 'agg_crimes_type_year.parquet', index=False)
agg_crime_hotspots.to_parquet(GOLD_DATA_DIR / 'agg_crime_hotspots.parquet', index=False)

print("✅ Agregações salvas:")
print(f"   📁 agg_crimes_area_period.parquet ({len(agg_crimes_area_period):,} registros)")
print(f"   📁 agg_crimes_type_year.parquet ({len(agg_crimes_type_year):,} registros)")
print(f"   📁 agg_crime_hotspots.parquet ({len(agg_crime_hotspots):,} registros)")

In [ ]:
# Salvar metadados
import json

gold_metadata = {
    'batch_id': BATCH_ID,
    'transformation_timestamp': datetime.now().isoformat(),
    'source_layer': 'silver',
    'target_layer': 'gold',
    'schema_type': 'star_schema',
    'fact_table': {
        'name': 'fato_crimes',
        'records': len(fato_crimes),
    },
    'dimensions': {
        'dim_date': len(dim_date),
        'dim_time': len(dim_time),
        'dim_area': len(dim_area),
        'dim_crime_type': len(dim_crime_type),
        'dim_weapon': len(dim_weapon),
        'dim_premise': len(dim_premise),
        'dim_victim': len(dim_victim),
    },
    'aggregations': {
        'agg_crimes_area_period': len(agg_crimes_area_period),
        'agg_crimes_type_year': len(agg_crimes_type_year),
        'agg_crime_hotspots': len(agg_crime_hotspots),
    },
}

metadata_path = GOLD_DATA_DIR / f'gold_metadata_{BATCH_ID}.json'
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(gold_metadata, f, indent=2)

print(f"✅ Metadados salvos: {metadata_path}")

## 7. Validação Final

In [ ]:
# Listar todos os arquivos Gold
print("📁 Arquivos na camada Gold:")
for file in sorted(os.listdir(GOLD_DATA_DIR)):
    filepath = GOLD_DATA_DIR / file
    size_kb = filepath.stat().st_size / 1024
    unit = 'KB'
    value = size_kb
    if size_kb > 1024:
        value = size_kb / 1024
        unit = 'MB'
    print(f"   📄 {file} ({value:.2f} {unit})")

In [ ]:
# Resumo final
print("=" * 70)
print("📊 RESUMO DA TRANSFORMAÇÃO - CAMADA GOLD")
print("=" * 70)
print(f"\n🔖 Batch ID: {BATCH_ID}")
print(f"📅 Data/Hora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n📊 MODELO DIMENSIONAL (Star Schema):")
print(f"\n   ⭐ TABELA FATO:")
print(f"      • fato_crimes: {len(fato_crimes):,} registros")

print(f"\n   📐 DIMENSÕES:")
print(f"      • dim_date: {len(dim_date):,} registros")
print(f"      • dim_time: {len(dim_time):,} registros")
print(f"      • dim_area: {len(dim_area):,} registros")
print(f"      • dim_crime_type: {len(dim_crime_type):,} registros")
print(f"      • dim_weapon: {len(dim_weapon):,} registros")
print(f"      • dim_premise: {len(dim_premise):,} registros")
print(f"      • dim_victim: {len(dim_victim):,} registros")

print(f"\n   📈 AGREGAÇÕES:")
print(f"      • agg_crimes_area_period: {len(agg_crimes_area_period):,} registros")
print(f"      • agg_crimes_type_year: {len(agg_crimes_type_year):,} registros")
print(f"      • agg_crime_hotspots: {len(agg_crime_hotspots):,} registros")

print(f"\n📁 Diretório: {GOLD_DATA_DIR}")
print("\n✅ Transformação Silver → Gold concluída com sucesso!")

## 📋 Resumo - Silver → Gold

### Modelo Dimensional Criado (Star Schema):

```
                    ┌─────────────────┐
                    │    dim_date     │
                    └────────┬────────┘
                             │
┌─────────────┐   ┌──────────┴──────────┐   ┌─────────────┐
│  dim_area   │───│     fato_crimes     │───│ dim_victim  │
└─────────────┘   └──────────┬──────────┘   └─────────────┘
                             │
┌─────────────┐              │              ┌─────────────┐
│dim_crime_type│─────────────┼──────────────│ dim_weapon  │
└─────────────┘              │              └─────────────┘
                    ┌────────┴────────┐
                    │   dim_premise   │
                    └─────────────────┘
```

### Transformações realizadas:
- ✅ Criação de Surrogate Keys para todas as dimensões
- ✅ Dimensão de Data com atributos temporais completos
- ✅ Dimensão de Tempo com período do dia e rush hour
- ✅ Dimensão de Área com classificação por região
- ✅ Dimensão de Tipo de Crime com categoria e severidade
- ✅ Dimensão de Arma com nível de letalidade
- ✅ Dimensão de Local com categoria e flag público/privado
- ✅ Dimensão de Vítima com perfil demográfico
- ✅ Tabela Fato com todas as foreign keys
- ✅ Tabelas agregadas para dashboards
- ✅ Persistência em formato Parquet

### Uso dos dados Gold:
Os dados estão prontos para:
- Ferramentas de BI (Power BI, Tableau, Metabase)
- Dashboards analíticos
- Relatórios executivos
- Machine Learning